# Análise de Vendas — Camada Gold

## Objetivo

Este notebook cria estruturas analíticas da camada Gold destinadas ao acompanhamento do desempenho comercial da **Distribuidora Horizonte**.

Os dados tratados da camada Silver são consolidados para responder perguntas como:

- Qual é o faturamento da empresa?
- Como as vendas evoluem ao longo do tempo?
- Qual é o lucro bruto?
- Qual é a margem obtida?
- Quantos pedidos são realizados?
- Quantos clientes compram em cada período?
- Qual é o ticket médio?
- Quais categorias geram mais receita?
- Quais vendedores apresentam melhor desempenho?
- As metas comerciais estão sendo atingidas?

## Indicadores principais

- faturamento;
- lucro bruto;
- margem percentual;
- quantidade vendida;
- número de pedidos;
- clientes ativos;
- produtos vendidos;
- ticket médio;
- percentual médio de desconto;
- crescimento mensal.

## Saídas

Este notebook cria as seguintes tabelas:

- `vendas_diarias`;
- `vendas_mensais`;
- `vendas_categoria_mensal`;
- `desempenho_vendedores_mensal`.

## Fluxo

Silver → Regras analíticas → Gold

## Regras

Foram criadas estruturas analíticas destinadas ao acompanhamento de vendas e desempenho comercial:

- vendas diárias;
- vendas mensais;
- vendas por categoria;
- desempenho mensal dos vendedores;
- atingimento de metas;
- ranking comercial.

A consistência entre os valores consolidados da camada Gold e os dados da camada Silver foi validada.
Essas tabelas serão utilizadas posteriormente na construção dos dashboards executivos e análises de negócio.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Origem: {schema_silver}")
print(f"Destino: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):
    """
    Salva um DataFrame como tabela Delta
    gerenciada na camada Gold.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
# ---------------------------------------------------------
# QUALITY GATE
# ---------------------------------------------------------

df_resultado_qualidade = (
    spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"resultado_qualidade"
    )
)


testes_criticos_reprovados = (

    df_resultado_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        f"Existem "
        f"{testes_criticos_reprovados} "
        f"testes críticos de qualidade reprovados. "
        f"A camada Gold não será processada."
    )


print(
    "Quality Gate aprovado. "
    "Processamento da camada Gold autorizado."
)

In [0]:
# ---------------------------------------------------------
# DADOS DE ORIGEM
# ---------------------------------------------------------

df_vendas = carregar_silver(
    "fato_vendas"
)

df_clientes = carregar_silver(
    "dim_cliente"
)

df_produtos = carregar_silver(
    "dim_produto"
)

df_vendedores = carregar_silver(
    "dim_vendedor"
)

df_metas = carregar_silver(
    "fato_metas_vendas"
)

In [0]:
data_referencia = (

    df_vendas

    .agg(
        F.max("data_venda")
        .alias("data_referencia")
    )

    .first()[
        "data_referencia"
    ]
)


print(
    f"Data de referência dos dados: "
    f"{data_referencia}"
)

In [0]:
# ---------------------------------------------------------
# BASE ANALÍTICA
# ---------------------------------------------------------

df_vendas_analiticas = (

    df_vendas.alias("v")

    .join(
        df_clientes.alias("c"),
        on="id_cliente",
        how="left"
    )

    .join(
        df_produtos.alias("p"),
        on="id_produto",
        how="left"
    )

    .join(
        df_vendedores.alias("vd"),
        on="id_vendedor",
        how="left"
    )

    .select(

        F.col("v.id_item_venda"),
        F.col("v.id_venda"),
        F.col("v.data_venda"),

        F.col("v.id_cliente"),
        F.col("c.nome_cliente"),
        F.col("c.segmento_cliente"),
        F.col("c.porte_cliente"),
        F.col("c.cidade"),
        F.col("c.estado"),

        F.col("v.id_produto"),
        F.col("p.nome_produto"),
        F.col("p.categoria"),
        F.col("p.subcategoria"),
        F.col("p.marca"),

        F.col("v.id_vendedor"),
        F.col("vd.nome_vendedor"),
        F.col("vd.regiao"),
        F.col("vd.nivel"),

        F.col("v.quantidade"),
        F.col("v.preco_unitario"),
        F.col("v.valor_bruto"),
        F.col("v.percentual_desconto"),
        F.col("v.valor_desconto"),
        F.col("v.valor_liquido"),
        F.col("v.custo_total"),
        F.col("v.lucro_bruto"),
        F.col("v.margem_percentual")
    )

    .withColumn(
        "ano",
        F.year("data_venda")
    )

    .withColumn(
        "mes",
        F.month("data_venda")
    )

    .withColumn(
        "mes_referencia",
        F.trunc(
            "data_venda",
            "month"
        )
    )

    .withColumn(
        "ano_mes",
        F.date_format(
            "data_venda",
            "yyyy-MM"
        )
    )
)

In [0]:
display(
    df_vendas_analiticas.limit(20)
)

In [0]:
# ---------------------------------------------------------
# VENDAS DIÁRIAS
# ---------------------------------------------------------

df_vendas_diarias = (

    df_vendas_analiticas

    .groupBy(
        "data_venda"
    )

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.round(
            F.sum(
                "valor_desconto"
            ),
            2
        ).alias(
            "valor_desconto"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "quantidade_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_ativos"
        ),

        F.countDistinct(
            "id_produto"
        ).alias(
            "produtos_vendidos"
        )
    )

    .withColumn(

        "ticket_medio",

        F.round(
            F.col("faturamento")
            /
            F.col("quantidade_pedidos"),
            2
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col("lucro_bruto")
            /
            F.col("faturamento")
            * 100,
            2
        )
    )

    .withColumn(
        "ano",
        F.year("data_venda")
    )

    .withColumn(
        "mes",
        F.month("data_venda")
    )

    .withColumn(
        "dia_semana",
        F.date_format(
            "data_venda",
            "EEEE"
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_vendas_diarias,
    "vendas_diarias"
)

In [0]:
# ---------------------------------------------------------
# VENDAS MENSAIS
# ---------------------------------------------------------

df_vendas_mensais = (

    df_vendas_analiticas

    .groupBy(
        "mes_referencia"
    )

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "valor_bruto"
            ),
            2
        ).alias(
            "faturamento_bruto"
        ),

        F.round(
            F.sum(
                "valor_desconto"
            ),
            2
        ).alias(
            "valor_desconto"
        ),

        F.round(
            F.sum(
                "custo_total"
            ),
            2
        ).alias(
            "custo_total"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "quantidade_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_ativos"
        ),

        F.countDistinct(
            "id_produto"
        ).alias(
            "produtos_vendidos"
        )
    )
)

In [0]:
df_vendas_mensais = (

    df_vendas_mensais

    .withColumn(

        "ticket_medio",

        F.round(
            F.col("faturamento")
            /
            F.col("quantidade_pedidos"),
            2
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col("lucro_bruto")
            /
            F.col("faturamento")
            * 100,
            2
        )
    )

    .withColumn(

        "percentual_desconto",

        F.round(
            F.col("valor_desconto")
            /
            F.col("faturamento_bruto")
            * 100,
            2
        )
    )
)

In [0]:
janela_mensal = (
    Window.orderBy(
        "mes_referencia"
    )
)


df_vendas_mensais = (

    df_vendas_mensais

    .withColumn(

        "faturamento_mes_anterior",

        F.lag(
            "faturamento"
        ).over(
            janela_mensal
        )
    )

    .withColumn(

        "crescimento_faturamento_percentual",

        F.when(
            F.col(
                "faturamento_mes_anterior"
            ).isNull(),

            None
        )

        .otherwise(

            F.round(

                (
                    F.col("faturamento")
                    -
                    F.col(
                        "faturamento_mes_anterior"
                    )
                )

                /

                F.col(
                    "faturamento_mes_anterior"
                )

                * 100,

                2
            )
        )
    )

    .withColumn(
        "ano",
        F.year(
            "mes_referencia"
        )
    )

    .withColumn(
        "mes",
        F.month(
            "mes_referencia"
        )
    )

    .withColumn(
        "ano_mes",
        F.date_format(
            "mes_referencia",
            "yyyy-MM"
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_vendas_mensais,
    "vendas_mensais"
)

In [0]:
display(
    df_vendas_mensais

    .select(
        "mes_referencia",
        "faturamento",
        "lucro_bruto",
        "margem_percentual",
        "quantidade_pedidos",
        "clientes_ativos",
        "ticket_medio",
        "crescimento_faturamento_percentual"
    )

    .orderBy(
        "mes_referencia"
    )
)

In [0]:
# ---------------------------------------------------------
# VENDAS POR CATEGORIA
# ---------------------------------------------------------

df_vendas_categoria_mensal = (

    df_vendas_analiticas

    .groupBy(
        "mes_referencia",
        "categoria"
    )

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "quantidade_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_ativos"
        ),

        F.countDistinct(
            "id_produto"
        ).alias(
            "produtos_vendidos"
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col("lucro_bruto")
            /
            F.col("faturamento")
            * 100,
            2
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_vendas_categoria_mensal,
    "vendas_categoria_mensal"
)

In [0]:
display(

    df_vendas_categoria_mensal

    .groupBy(
        "categoria"
    )

    .agg(

        F.round(
            F.sum("faturamento"),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum("lucro_bruto"),
            2
        ).alias(
            "lucro_bruto"
        )
    )

    .orderBy(
        F.desc("faturamento")
    )
)

In [0]:
# ---------------------------------------------------------
# DESEMPENHO DOS VENDEDORES
# ---------------------------------------------------------

df_desempenho_vendas = (

    df_vendas_analiticas

    .groupBy(
        "mes_referencia",
        "id_vendedor",
        "nome_vendedor",
        "regiao",
        "nivel"
    )

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "quantidade_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_positivados"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        )
    )

    .withColumn(

        "ticket_medio",

        F.round(
            F.col("faturamento")
            /
            F.col("quantidade_pedidos"),
            2
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col("lucro_bruto")
            /
            F.col("faturamento")
            * 100,
            2
        )
    )
)

In [0]:
df_desempenho_vendedores = (

    df_desempenho_vendas.alias("v")

    .join(

        df_metas.alias("m"),

        on=[
            "mes_referencia",
            "id_vendedor"
        ],

        how="left"
    )

    .select(

        "mes_referencia",
        "id_vendedor",
        "nome_vendedor",
        "regiao",
        "nivel",

        "faturamento",
        "lucro_bruto",
        "margem_percentual",

        "quantidade_pedidos",
        "clientes_positivados",
        "quantidade_vendida",
        "ticket_medio",

        F.col(
            "m.valor_meta"
        ).alias(
            "valor_meta"
        ),

        F.col(
            "m.meta_clientes"
        ).alias(
            "meta_clientes"
        )
    )
)

In [0]:
df_desempenho_vendedores = (

    df_desempenho_vendedores

    .withColumn(

        "atingimento_meta_percentual",

        F.round(
            F.col("faturamento")
            /
            F.col("valor_meta")
            * 100,
            2
        )
    )

    .withColumn(

        "atingimento_meta_clientes_percentual",

        F.round(
            F.col("clientes_positivados")
            /
            F.col("meta_clientes")
            * 100,
            2
        )
    )

    .withColumn(

        "status_meta",

        F.when(
            F.col(
                "atingimento_meta_percentual"
            ) >= 100,

            F.lit(
                "Meta atingida"
            )
        )

        .otherwise(
            F.lit(
                "Meta não atingida"
            )
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_desempenho_vendedores,
    "desempenho_vendedores_mensal"
)

In [0]:
janela_ranking = (

    Window

    .partitionBy(
        "mes_referencia"
    )

    .orderBy(
        F.desc(
            "faturamento"
        )
    )
)

In [0]:
df_desempenho_vendedores = (

    df_desempenho_vendedores

    .withColumn(

        "ranking_faturamento",

        F.dense_rank()
        .over(
            janela_ranking
        )
    )
)

In [0]:
salvar_gold(
    df_desempenho_vendedores,
    "desempenho_vendedores_mensal"
)

In [0]:
ultimo_mes = (

    df_desempenho_vendedores

    .agg(
        F.max(
            "mes_referencia"
        )
        .alias(
            "ultimo_mes"
        )
    )

    .first()[
        "ultimo_mes"
    ]
)


print(
    f"Último mês disponível: {ultimo_mes}"
)

In [0]:
display(

    df_desempenho_vendedores

    .filter(
        F.col(
            "mes_referencia"
        ) == ultimo_mes
    )

    .select(
        "ranking_faturamento",
        "nome_vendedor",
        "regiao",
        "faturamento",
        "valor_meta",
        "atingimento_meta_percentual",
        "clientes_positivados",
        "ticket_medio",
        "margem_percentual",
        "status_meta"
    )

    .orderBy(
        "ranking_faturamento"
    )
)

In [0]:
df_kpis_gerais = (

    df_vendas_analiticas

    .agg(

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento_total"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto_total"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "total_pedidos"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "clientes_com_compra"
        ),

        F.countDistinct(
            "id_produto"
        ).alias(
            "produtos_vendidos"
        ),

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_vendida"
        )
    )

    .withColumn(

        "ticket_medio",

        F.round(
            F.col(
                "faturamento_total"
            )
            /
            F.col(
                "total_pedidos"
            ),
            2
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col(
                "lucro_bruto_total"
            )
            /
            F.col(
                "faturamento_total"
            )
            * 100,
            2
        )
    )
)


display(
    df_kpis_gerais
)

In [0]:
display(

    df_vendas_analiticas

    .groupBy(
        "mes"
    )

    .agg(
        F.round(
            F.avg(
                "valor_liquido"
            ),
            2
        ).alias(
            "valor_medio_item"
        ),

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento_total"
        )
    )

    .orderBy(
        "mes"
    )
)

In [0]:
display(
    spark.sql(
        f"""
        SHOW TABLES
        IN `{catalogo_atual}`.`{schema_gold}`
        """
    )
)

In [0]:
comentarios_gold = {

    "vendas_diarias":
        "Indicadores comerciais consolidados por dia.",

    "vendas_mensais":
        "Indicadores comerciais consolidados por mês, incluindo crescimento e rentabilidade.",

    "vendas_categoria_mensal":
        "Indicadores mensais de vendas e rentabilidade por categoria de produto.",

    "desempenho_vendedores_mensal":
        "Indicadores mensais de desempenho comercial dos vendedores e atingimento de metas."
}


for tabela, comentario in comentarios_gold.items():

    spark.sql(
        f"""
        COMMENT ON TABLE
        `{catalogo_atual}`.`{schema_gold}`.`{tabela}`
        IS '{comentario}'
        """
    )


print(
    "Descrições adicionadas às tabelas Gold."
)

In [0]:
faturamento_silver = (

    df_vendas

    .agg(
        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "faturamento"
        )
    )

    .first()[
        "faturamento"
    ]
)


faturamento_gold = (

    df_vendas_mensais

    .agg(
        F.round(
            F.sum(
                "faturamento"
            ),
            2
        ).alias(
            "faturamento"
        )
    )

    .first()[
        "faturamento"
    ]
)


print(
    f"Faturamento Silver: "
    f"R$ {faturamento_silver:,.2f}"
)

print(
    f"Faturamento Gold:   "
    f"R$ {faturamento_gold:,.2f}"
)

In [0]:
diferenca = abs(
    float(faturamento_silver)
    -
    float(faturamento_gold)
)


if diferenca > 0.05:

    raise Exception(
        "O faturamento consolidado da Gold "
        "não corresponde ao faturamento da Silver."
    )


print(
    "Validação concluída: "
    "Gold e Silver apresentam o mesmo faturamento."
)